# Inspect Aggregated Daymet SWE (NA region)

HRU-level inspection of Daymet V4 R1 daily snow water equivalent. Companion to `aggregate/daymet.py`.

Source (see `catalog/sources.yml` → `daymet`):

- Daymet V4 R1, North America zarr (`daymet_na.zarr`), daily 1 km on a **Lambert Conformal Conic** grid. `swe` in `kg m-2` (≡ mm of water at density 1000), `cell_methods: "time: point"` (instantaneous, not accumulated).

Per the repo's transformation policy (`docs/architecture/transformation-pipeline.md`): the aggregator preserves Daymet's native `kg m-2` unit; the `÷ 25.4` rescale to **inches** (PRMS `pkwater_equiv`) is a linear conversion that happens **post-aggregation** — this notebook applies it inline for display, and `targets/swe.py` will apply it for the final calibration target.

Region encoding: the aggregator emits `daymet_na_<year>_agg.nc` (region in the filename) so HI/PR can land alongside NA later without renaming existing NA outputs. This notebook reads `region="na"`.

## Per-target conventions in this notebook

- The aggregated NC carries `swe` in its **native `kg m-2` (≡ mm)** units. Daymet's NA grid is bounded to the continent; HRUs whose footprint extends past the Daymet domain come out NaN (geometric partial coverage).
- This notebook converts to **inches** (÷ 25.4) for the PRMS-units sanity check; the underlying values remain in mm.
- SWE is **instantaneous** (`cell_methods: "time: point"`) — daily snapshots, never accumulated. A monthly aggregate is the mean over time, not a sum.
- Single-source-in-isolation view here: this notebook is one piece of the four-source SWE target (Daymet, SNODAS, ERA5-Land `sd`, Margulis WUS-SR). Cross-source comparison lives in `inspect_aggregated_swe.ipynb` (lands with the target-builder PR).
- **Compare with `inspect_aggregated_snodas.ipynb`** for the same TARGET_YEAR / TARGET_MONTH. The two sources should agree on the broad geographic SWE pattern; persistent disagreement in mountain HRUs is real (different physics / observation networks) and informs the eventual SWE-target multi-source min/max.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/aggregated/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = False
_helpers.PROJECT = PROJECT_DIR.name

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

SOURCE_KEY = "daymet"
REGION = "na"
VAR = "swe"
# Mid-archive year with a typical CONUS winter pack; matches the
# default in inspect_aggregated_snodas.ipynb so the two notebooks are
# directly comparable.
TARGET_YEAR = 2010
TARGET_DAY = "2010-02-15"
TARGET_MONTH = 2  # February
TIME_SERIES_YEARS = range(2003, 2010)
# kg m-2 ≡ mm water; ÷ 25.4 → inches (PRMS pkwater_equiv).
MM_PER_INCH = 25.4

REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Northern Rockies (MT/WY)": (-110.5, 45.5),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated dataset

Opens `<project>/data/aggregated/daymet/daymet_na_<TARGET_YEAR>_agg.nc` via the region-aware `discover_aggregated(..., region="na")` helper. If the year hasn't been aggregated yet, the cell prints a clear skip message and the remaining cells fall through gracefully.

In [ ]:
paths = discover_aggregated(project_dir, SOURCE_KEY, region=REGION)
if paths is None:
    print(
        f"SKIP {SOURCE_KEY} ({REGION}): no aggregated NCs at "
        f"{project_dir}/data/aggregated/{SOURCE_KEY}/daymet_{REGION}_*_agg.nc. "
        f"Run `pixi run nhf-targets agg daymet --project-dir {project_dir} "
        f"--period 1980/2024` first."
    )
    ds = None
else:
    ds = open_year(project_dir, SOURCE_KEY, TARGET_YEAR, region=REGION)
    units = unit_from_catalog(SOURCE_KEY, VAR)
    values_t0 = ds[VAR].isel(time=0).to_pandas()
    print(
        f"Daymet {REGION.upper()}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get(fabric_cfg['id_col'], 'unknown')} | "
        f"NaN HRUs (t=0): {nan_hru_count(values_t0)} | "
        f"catalog units: {units} | "
        f"daymet_region attr: {ds.attrs.get('daymet_region', '<missing>')}"
    )
    display(ds)

## Daily snapshot + monthly mean (mm and inches)

Four HRU choropleths: native `kg m-2` (≡ mm) on top, PRMS-units (`÷ 25.4` → inches) on the bottom; daily snapshot on the left, monthly mean on the right. NaN HRUs are light grey — typically a thin band along the Atlantic/Pacific coasts and the Canadian/Mexican borders where the HRU footprint extends past the Daymet NA domain.

In [ ]:
if ds is None:
    print("No aggregated Daymet; skipping panels.")
else:
    da_day_mm = ds[VAR].sel(time=TARGET_DAY, method="nearest")
    monthly_mm = (
        ds[VAR]
        .sel(
            time=slice(
                pd.Timestamp(year=TARGET_YEAR, month=TARGET_MONTH, day=1),
                pd.Timestamp(year=TARGET_YEAR, month=TARGET_MONTH, day=1)
                + pd.offsets.MonthEnd(0),
            )
        )
        .mean("time", skipna=True)
        .to_pandas()
    )
    day_mm = da_day_mm.to_pandas()
    monthly_in = monthly_mm / MM_PER_INCH

    vmax_mm = float(np.nanpercentile(day_mm.dropna(), 95))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), squeeze=False)
    plot_hru_choropleth(
        axes[0, 0], fabric, day_mm,
        vmin=0, vmax=vmax_mm, cmap="Blues",
        title=f"Daily SWE\n{TARGET_DAY}",
        units="mm (kg m-2)",
    )
    plot_hru_choropleth(
        axes[0, 1], fabric, monthly_mm,
        vmin=0, vmax=vmax_mm, cmap="Blues",
        title=f"Monthly mean SWE\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="mm (kg m-2)",
    )
    plot_hru_choropleth(
        axes[1, 0], fabric, day_mm / MM_PER_INCH,
        vmin=0, vmax=vmax_mm / MM_PER_INCH, cmap="Blues",
        title=f"Daily SWE (PRMS units)\n{TARGET_DAY}",
        units="inches",
    )
    plot_hru_choropleth(
        axes[1, 1], fabric, monthly_in,
        vmin=0, vmax=vmax_mm / MM_PER_INCH, cmap="Blues",
        title=f"Monthly mean SWE (PRMS units)\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="inches",
    )
    fig.suptitle(
        "Daymet NA SWE on HRU fabric — aggregator preserves native kg m-2; PRMS ÷ 25.4 rescale here",
        fontsize=12, y=1.00,
    )
    plt.tight_layout()
    save_figure(fig, "daymet_na_native_units_map")
    plt.show()

### Reading the panels

- **Top row — daily snapshot vs. monthly mean (mm).** The daily snapshot is dominated by the highest-elevation HRUs (Cascades, Sierras, Rockies, Adirondacks). The monthly mean is a smoothed version; broad geographic pattern should match the daily.
- **Bottom row — same data in PRMS units (inches).** The unit the SWE target ultimately emits. Visual pattern identical to the top row (linear rescale); only the numeric scale changes.
- **NaN HRUs.** Coastal HRUs whose pixel footprint extends past the Daymet NA grid come out NaN. Compare with SNODAS — Daymet's land mask differs (Daymet covers all of NA including Canada/Mexico), so the NaN pattern at borders will be smaller than SNODAS's CONUS-only mask.
- **Red flags.** A unimodal map with no snow in the Rockies in February is a missed `_FillValue` decode or unit-conversion error. A wildly compressed colour scale usually means LCC-coord reprojection in gdptools picked up only a sliver of the source grid — verify by checking `vmax_mm` in the previous cell; values in the tens-to-hundreds mm range are sane, single-digit values are suspect.

In [ ]:
if ds is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    monthly_finite_mm = monthly_mm.dropna()
    ax.hist(monthly_finite_mm, bins=60, histtype="step", linewidth=2, density=True)
    ax.set_xlabel("SWE (mm, monthly mean)")
    ax.set_ylabel("Density")
    ax.set_title(
        f"HRU-level monthly-mean SWE distribution — Daymet NA "
        f"{TARGET_YEAR}-{TARGET_MONTH:02d}"
    )
    save_figure(fig, "daymet_na_histogram")
    plt.show()
    print(
        f"monthly mean SWE summary (mm): "
        f"min={monthly_finite_mm.min():.2f}, "
        f"median={monthly_finite_mm.median():.2f}, "
        f"mean={monthly_finite_mm.mean():.2f}, "
        f"95th={np.nanpercentile(monthly_finite_mm, 95):.2f}, "
        f"max={monthly_finite_mm.max():.2f}"
    )

### Reading the SWE histogram

Expected shape for a winter month is **right-skewed with a heavy zero peak** (same as SNODAS):

- Tall spike at **0 mm** — most HRUs (lower elevation / southern CONUS) carry essentially no snow.
- Long right tail — mountain HRUs reach hundreds of mm; the deepest Sierra and PNW maritime HRUs can exceed 1000 mm.
- Near-continuous distribution between, populated by mid-elevation and northern HRUs.

Red flags:

- **No HRUs above ~200 mm in February.** Either Daymet's deep-pack regions weren't covered by the fabric, or the LCC reprojection went wrong (less likely — gdptools handles this).
- **Spike at a non-zero round value** — fill-value decode failure (Daymet typically encodes fills as NaN-float, not sentinel ints, but worth checking).
- **Unimodal Gaussian in winter** would be wrong — SWE is fundamentally bimodal (snowy vs not).

In [ ]:
if ds is not None:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    ds_range = open_year_range(
        project_dir, SOURCE_KEY, TIME_SERIES_YEARS, region=REGION
    )
    try:
        id_dim = fabric_cfg["id_col"]
        swe_full_mm = (
            ds_range[VAR]
            .sel({id_dim: list(rep_hrus.values())})
            .load()
        )
    finally:
        ds_range.close()

    swe_full_in = swe_full_mm / MM_PER_INCH

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        ax.plot(
            swe_full_in.time,
            swe_full_in.sel({id_dim: hru_id}).values,
            linewidth=0.6,
        )
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("SWE (inches)")
        ax.grid(alpha=0.3)
    fig.suptitle(
        f"Daymet NA SWE at representative HRUs — {min(TIME_SERIES_YEARS)}–"
        f"{max(TIME_SERIES_YEARS)} (daily, PRMS units)"
    )
    plt.tight_layout()
    save_figure(fig, "daymet_na_time_series")
    plt.show()

### Representative-HRU seasonality check

Each panel is a multi-year daily SWE time series at one HRU, in PRMS units (inches). Expected signatures should match the SNODAS notebook for the same HRUs; Daymet uses a different model + data assimilation pipeline so absolute magnitudes will differ, but seasonal shapes should agree.

- **Olympic Peninsula / Northern Rockies** — Strong winter peaks (Dec–Apr) reaching tens of inches; deepest maritime snowpacks can exceed 100 in mid-winter. Slow spring melt curve through May–Jun, near zero Jul–Oct.
- **Iowa cropland** — Sharper winter peaks (Jan–Feb) under ~10 inches, rapid early-spring melt, zero pack May–Oct.
- **Southern Appalachians** — Short snow pulses lasting days to a week or two, small amplitude (< 5 inches typically). Trace should look spiky.

**Cross-check.** Plot the same HRUs from `inspect_aggregated_snodas.ipynb` side-by-side and compare:
- **Mountain HRUs:** Daymet often runs higher than SNODAS in peak SWE (Daymet's gridded reconstruction can over-estimate in complex terrain where SNODAS's assimilation pulls toward sparse ground-truth).
- **Lowland HRUs:** the two sources usually agree closely on the magnitude and timing of snow events.
- **Persistent disagreement** in any one region is informative for the eventual SWE target's multi-source min/max — it means the bound there is wider than the single-source estimate.

In [ ]:
if ds is not None:
    n_nan = nan_hru_count(monthly_mm)
    print(
        f"Daymet NA {TARGET_YEAR}-{TARGET_MONTH:02d}: "
        f"{n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)"
    )
    fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
    plot_nan_hrus(
        axes[0, 0],
        fabric,
        monthly_mm,
        title=(
            f"NaN HRUs (red) — {n_nan} of {len(fabric)} "
            f"(monthly mean, {TARGET_YEAR}-{TARGET_MONTH:02d})"
        ),
    )
    fig.suptitle(
        "Coverage gaps — Daymet NA domain edge; NN-fill happens at target stage",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, "daymet_na_coverage")
    plt.show()

In [ ]:
if ds is not None:
    # Magnitude sanity check: HRU-area-weighted monthly mean (mm) over
    # the fabric. Daymet's CONUS-mean Feb SWE lands in the tens-of-mm
    # range, similar to SNODAS. Memory: feedback_validate_magnitudes —
    # an order-of-magnitude miss is a smoking gun for a missed _FillValue
    # decode, LCC reprojection failure, or unit conversion.
    agg_mean_mm = area_weighted_mean(monthly_mm, fabric)
    agg_mean_in = (
        agg_mean_mm / MM_PER_INCH if agg_mean_mm == agg_mean_mm else float("nan")
    )
    print(
        f"Daymet NA {TARGET_YEAR}-{TARGET_MONTH:02d} HRU-area-weighted mean: "
        f"{agg_mean_mm:.2f} mm  ({agg_mean_in:.3f} inches)"
    )
    print(
        "Expected order of magnitude for CONUS winter peak: tens of mm "
        "(typical February ≈ 20–40 mm); compare with the same line in "
        "inspect_aggregated_snodas.ipynb. An order-of-magnitude miss is "
        "a smoking gun for a missed _FillValue decode, LCC reprojection "
        "failure, or unit conversion error."
    )

## Why this is a single-source diagnostic, not a cross-source comparison

Daymet is one of four sources for the SWE calibration target. This notebook is checking that the Daymet-NA aggregation is sane on its own; the cross-source view (with SNODAS, ERA5-Land `sd`, and Margulis WUS-SR alongside) lives in the to-be-written `inspect_aggregated_swe.ipynb`.

- **Lambert Conformal Conic source grid.** Unlike every other source in this pipeline (which are WGS84 lat/lon), Daymet's zarr is in a projected LCC grid with `x` / `y` in metres. The aggregator decodes the CRS from the zarr's `lambert_conformal_conic` grid-mapping variable via `pyproj.CRS.from_cf` and passes the resulting WKT to gdptools `WeightGen`. If the LCC reprojection silently fails (e.g. wrong projection params, mismatched datum), this notebook's spatial maps are how you'd notice — the snow pattern would be displaced, mirrored, or compressed.
- **No quality gate at aggregate time.** Daymet's `swe` is a continuous float; xarray decodes any `_FillValue` to NaN at open time. The aggregator uses `stat_method="mean"` with no `pre_aggregate_hook`; HRUs with NaN pixels propagate to NaN at the HRU level (honest geometric partial coverage).
- **Units pass through.** The aggregator preserves native `kg m-2`. The `÷ 25.4` rescale to inches is the notebook's / target's job.
- **Time is point, not sum.** SWE is instantaneous; a monthly aggregate is the mean over daily snapshots.
- **NA region only in this build.** The aggregator's `--region` flag accepts `na` (default); `hi`/`pr` raise `NotImplementedError` until those fabrics need them (issue #101).

**Calibration target implication.** The SWE target (`targets/swe.py`, separate PR) reads this aggregated NC, combines it with the other three SWE sources via NaN-aware min/max, optionally NN-fills the combined bound on HRUs that were NaN everywhere, and emits the final calibration target as PRMS `pkwater_equiv` in inches. What this notebook checks is that the Daymet-NA-side input to that pipeline is sane on its own.

In [ ]:
if ds is not None:
    ds.close()